# 项目结构总览

```
app/
├── main.py                    # app 实例创建、lifespan、挂载路由
├── core/
│   ├── __init__.py
│   └── database.py            # 数据库引擎、建表、session 生成器
├── models/
│   ├── __init__.py
│   ├── user.py                # SQLModel 数据模型
│   ├── request.py             # 请求体 Pydantic 模型
│   └── response.py            # 响应体 Pydantic 模型
├── routers/
│   ├── __init__.py
│   └── user.py                # 用户路由（接收请求 → 调用 service → 返回响应）
└── services/
    ├── __init__.py
    └── user.py                # 用户业务逻辑 + 密码工具
```

**分层职责：**
- **router** — 只负责接收 HTTP 请求、调用 service、返回响应
- **service** — 业务逻辑、数据库操作、密码工具
- **model** — 数据模型定义

# 第一步：数据库配置 — `app/core/database.py`

## 逐行解释

**`load_dotenv()`** — 读取项目根目录的 `.env` 文件，把里面的键值对注入到环境变量中。这样你就不用硬编码数据库密码。

**`DATABASE_URL`** — 拼接出一个 SQLAlchemy 连接字符串，格式是：
```
mysql+pymysql://用户名:密码@主机:端口/数据库名
```

**`engine = create_engine(...)`** — 创建数据库引擎。它是连接池的入口，管理着与数据库的连接。`echo=True` 表示会把执行的 SQL 语句打印到终端，方便调试。注意：此时**并没有真正连接数据库**，只是准备好了连接配置。

**`create_db_and_tables()`** — 读取所有继承自 `SQLModel` 的模型类，执行 `CREATE TABLE IF NOT EXISTS`。需要在 FastAPI 启动时调用一次。

**`get_session()`** — 这是一个生成器函数，用 `with Session(engine)` 创建一个数据库会话，用完自动关闭。`yield` 让它成为 FastAPI 的依赖注入函数，配合 `Depends(get_session)` 使用，FastAPI 会在请求结束时自动关闭 session。

In [ ]:
# app/core/database.py
import os

from dotenv import load_dotenv
from sqlmodel import SQLModel, Session, create_engine

load_dotenv()

DATABASE_URL = (
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT', '3306')}/{os.getenv('DB_NAME')}"
)

engine = create_engine(DATABASE_URL, echo=True)


def create_db_and_tables():
    SQLModel.metadata.create_all(engine)


def get_session():
    with Session(engine) as session:
        yield session

# 第二步：定义数据模型

项目有三种模型，各司其职：

| 文件 | 用途 | 基类 |
|------|------|------|
| `models/user.py` | 数据库表模型（ORM） | `SQLModel, table=True` |
| `models/request.py` | 请求体校验 | `BaseModel`（纯 Pydantic） |
| `models/response.py` | 统一响应格式 | `BaseModel`（纯 Pydantic） |

# 数据库表模型 — app/models/user.py
from sqlmodel import SQLModel, Field


class User(SQLModel, table=True):   # table=True 表示这是一个数据库表
    id: int | None = Field(default=None, primary_key=True)
    name: str
    password: str                   # 存储 bcrypt 哈希后的密码
    email: str

In [ ]:
# 请求体模型 — app/models/request.py
from pydantic import BaseModel


class UserCreateRequest(BaseModel):
    name: str
    password: str
    email: str


class UserLoginRequest(BaseModel):
    name: str
    password: str


class PasswordUpdateRequest(BaseModel):
    id: int
    old_password: str
    new_password: str


class UserUpdateRequest(BaseModel):
    id: int
    name: str

# 响应体模型 — app/models/response.py
from typing import Any
from pydantic import BaseModel


class APIResponse(BaseModel):
    code: int = 0
    message: str = "success"
    data: Any = None


class UserResponse(BaseModel):
    name: str
    email: str

# 第三步：Service 层 — `app/services/user.py`

业务逻辑全部集中在这里，包括密码工具和所有数据库操作。
Router 层只负责接收请求、调用 service、返回响应，不包含任何业务逻辑。

# 密码工具 — passlib + bcrypt
from passlib.context import CryptContext

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")


def hash_password(password: str) -> str:
    return pwd_context.hash(password)


def verify_password(plain_password: str, hashed_password: str) -> bool:
    return pwd_context.verify(plain_password, hashed_password)

In [ ]:
# 业务逻辑 — app/services/user.py（续）
# 所有 service 函数接收 session 作为参数，操作数据库，返回 APIResponse
from sqlmodel import Session, select
from app.models.user import User
from app.models.response import APIResponse, UserResponse

### Create — 注册用户

```python
# select(User).where(...) — 构建条件查询
# session.exec(statement).first() — 执行查询，取第一条
# session.add(user) — 将对象加入会话
# session.commit() — 提交事务，真正写入数据库
# session.refresh(user) — 刷新，拿到数据库生成的 id 等字段
```

In [ ]:
def register(session: Session, name: str, password: str, email: str) -> APIResponse:
    # 检查用户名是否已存在
    statement = select(User).where(User.name == name)
    existing = session.exec(statement).first()
    if existing is not None:
        return APIResponse(code=1, message="username already exists")

    # 创建新用户，密码用 bcrypt 哈希后存储
    user = User(
        name=name,
        password=hash_password(password),
        email=email,
    )
    session.add(user)       # 加入会话
    session.commit()        # 提交事务
    session.refresh(user)   # 拿到数据库生成的 id
    return APIResponse(
        data=UserResponse(name=user.name, email=user.email),
    )

### Read — 读取用户

```python
# session.get(Model, 主键) — 按主键查一条，最快捷
# session.exec(select(Model).where(...)) — 按条件查询，更灵活
```

In [ ]:
def login(session: Session, name: str, password: str) -> APIResponse:
    # 条件查询：按用户名查找
    statement = select(User).where(User.name == name)
    user = session.exec(statement).first()
    if user is None:
        return APIResponse(code=1, message="user not found")
    if not verify_password(password, user.password):
        return APIResponse(code=2, message="wrong password")
    return APIResponse(
        data=UserResponse(name=user.name, email=user.email),
    )


def read_user(session: Session, user_id: int) -> APIResponse:
    # 主键查询：直接按 id 取
    user = session.get(User, user_id)
    if user is None:
        return APIResponse(code=1919810, message="user not found")
    return APIResponse(
        data=UserResponse(name=user.name, email=user.email)
    )

### Update — 更新用户

```python
# 修改模型字段 → session.add() → session.commit()
# 如果需要拿更新后的值，再加 session.refresh()
```

In [ ]:
def update_password(session: Session, user_id: int, old_password: str, new_password: str) -> APIResponse:
    user = session.get(User, user_id)
    if user is None:
        return APIResponse(code=1919810, message="user not found")
    if not verify_password(old_password, user.password):
        return APIResponse(code=2, message="wrong password")
    user.password = hash_password(new_password)   # 修改字段
    session.add(user)                             # 加入会话
    session.commit()                              # 提交事务
    return APIResponse(message="password updated")


def update_name(session: Session, user_id: int, name: str) -> APIResponse:
    user = session.get(User, user_id)
    if user is None:
        return APIResponse(code=1919810, message="user not found")
    # 检查新名字是否已被占用
    statement = select(User).where(User.name == name)
    existing = session.exec(statement).first()
    if existing is not None:
        return APIResponse(code=1, message="username already exists")
    user.name = name                              # 修改字段
    session.add(user)                             # 加入会话
    session.commit()                              # 提交事务
    session.refresh(user)                         # 刷新拿到最新值
    return APIResponse(data=UserResponse(name=user.name, email=user.email))

### Delete — 删除用户

In [ ]:
def delete_user(session: Session, user_id: int) -> APIResponse:
    user = session.get(User, user_id)
    if user is None:
        return APIResponse(code=1919810, message="user not found")
    session.delete(user)    # 标记删除
    session.commit()        # 提交事务
    return APIResponse(message="user deleted")

# 第四步：Router 层 — `app/routers/user.py`

路由函数只做三件事：**接收请求 → 调用 service → 返回响应**。
用 `Depends(get_session)` 注入数据库 session，然后传给 service 层。

In [ ]:
# app/routers/user.py
from fastapi import APIRouter, Depends, status
from sqlmodel import Session

from app.core import get_session
from app.models.response import APIResponse
from app.models.request import UserCreateRequest, UserLoginRequest, PasswordUpdateRequest, UserUpdateRequest
from app.services import user as user_service

# prefix 给所有路由加上 /user 前缀，tags 用于 /docs 页面分组
router = APIRouter(prefix="/user", tags=["user"])


@router.get("/{user_id}", status_code=status.HTTP_200_OK, response_model=APIResponse)
def read_user(user_id: int, session: Session = Depends(get_session)):
    return user_service.read_user(session, user_id)


@router.post("/register", status_code=status.HTTP_201_CREATED, response_model=APIResponse)
def register(req: UserCreateRequest, session: Session = Depends(get_session)):
    return user_service.register(session, req.name, req.password, req.email)


@router.post("/login", status_code=status.HTTP_200_OK, response_model=APIResponse)
def login(req: UserLoginRequest, session: Session = Depends(get_session)):
    return user_service.login(session, req.name, req.password)


@router.put("/password", status_code=status.HTTP_200_OK, response_model=APIResponse)
def update_password(req: PasswordUpdateRequest, session: Session = Depends(get_session)):
    return user_service.update_password(session, req.id, req.old_password, req.new_password)


@router.put("/name", status_code=status.HTTP_200_OK, response_model=APIResponse)
def update_name(req: UserUpdateRequest, session: Session = Depends(get_session)):
    return user_service.update_name(session, req.id, req.name)


@router.delete("/{user_id}", status_code=status.HTTP_200_OK, response_model=APIResponse)
def delete_user(user_id: int, session: Session = Depends(get_session)):
    return user_service.delete_user(session, user_id)

# 第五步：App 入口 — `app/main.py`

只保留：lifespan（启动时建表）、app 实例、根路由、`include_router` 挂载。

In [ ]:
# app/main.py
from contextlib import asynccontextmanager
from fastapi import FastAPI, status

from app.core import create_db_and_tables
from app.models.response import APIResponse
from app.routers import user as user_router


@asynccontextmanager
async def lifespan(app):
    create_db_and_tables()   # 启动时建表
    yield

app = FastAPI(lifespan=lifespan)

app.include_router(user_router.router)   # 挂载用户路由


@app.get("/", status_code=status.HTTP_200_OK)
def read_root():
    return APIResponse(data={"content": "Hello, world!"})

# 整体请求流程

```
HTTP 请求
  → FastAPI 匹配路由（app/routers/user.py）
  → Depends(get_session) 创建数据库 session
  → 路由函数调用 service 层（app/services/user.py）
  → service 操作数据库（session.add / commit / get / exec / delete）
  → service 返回 APIResponse
  → 路由函数将 APIResponse 返回给客户端
  → 请求结束，session 自动关闭
```

## 核心 ORM 操作速查

| 操作 | 方法 |
|------|------|
| 新增 | `session.add(obj)` + `session.commit()` + `session.refresh(obj)` |
| 条件查询 | `session.exec(select(Model).where(...)).first()` / `.all()` |
| 主键查询 | `session.get(Model, 主键)` |
| 更新 | 修改字段 → `session.add(obj)` + `session.commit()` |
| 删除 | `session.delete(obj)` + `session.commit()` |